In [1]:
import torch
import torchvision.transforms as transforms
from torchvision.io import read_image

import importlib
import funcs
from sklearn.metrics import confusion_matrix
import numpy as np
import time

Torch device: cuda
NVIDIA GeForce RTX 3070 Laptop GPU


In [7]:
from torchvision.models import resnext50_32x4d, ResNeXt50_32X4D_Weights
model = resnext50_32x4d(weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V2)

Downloading: "https://download.pytorch.org/models/resnext50_32x4d-1a0047aa.pth" to /home/thomas/.cache/torch/hub/checkpoints/resnext50_32x4d-1a0047aa.pth
100.0%


In [8]:
ResNeXt50_32X4D_Weights.IMAGENET1K_V2.transforms


for name, module in model.named_children():
    print(name, module)

conv1 Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
bn1 BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
relu ReLU(inplace=True)
maxpool MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
layer1 Sequential(
  (0): Bottleneck(
    (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv3): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (downsample): Sequential(
      (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm

In [ ]:
num_features = model.fc.in_features
n_subend = 128

model.fc = nn.Linear(num_features, num_classes)

In [3]:
funcs.get_conv_resize(22, 3, 2, 0)


10

In [ ]:
image = read_image('data/source/images/n02086646-Blenheim_spaniel/n02086646_3.jpg')
print(image.shape)

In [ ]:
checkpoint = torch.load('data/models/initialized_first_reduced_model')

In [ ]:
import signal
import time

KEEPRUN = True

def handle_interrupt(signal, frame):
    # Perform actions or cleanup tasks when Ctrl+C is pressed
    print("Interrupt signal received. Stopping the function or performing cleanup.")
    global KEEPRUN
    KEEPRUN = False

# Register the signal handler function
signal.signal(signal.SIGINT, handle_interrupt)

# Your function code here
# ...

# Infinite loop example
while KEEPRUN:
    print('computing...')
    time.sleep(1.)


In [ ]:
checkpoint['model_state_dict']

In [ ]:
from torch.nn.functional import normalize
means = image.float().mean((1,2))
image2 = image - means.reshape(3,1,1)
var = (image2**2).sum((1,2))
print('means:', image2.mean((1,2)))
print('means:', var)

# image_normed = transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                                     std=[0.229, 0.224, 0.225])(image.float())
image_normed = normalize(image.float(), p=2, dim=0)
print('\nimage normed:', image_normed.shape)
print(image_normed[0].min(), image_normed[1].min(), image_normed[2].min())
print(image_normed[0].max(), image_normed[1].max(), image_normed[2].max())
normed_means = image_normed.mean((1,2))
print('normed means:', normed_means)
print('normed std:', np.sqrt(((image_normed-normed_means.reshape(3,1,1))**2).sum((1,2))))

print('image')
for t in image.float():
    mean, std, var = torch.mean(t), torch.std(t), torch.var(t)
    print(mean, std, var)

image_f = image.float()
# image_normed = image_f - image_f.mean((1,2)).reshape(-1,1,1)
avg = image_f.mean((1,2))
n = image.shape[1] * image.shape[2]
# var = (image_normed**2).sum((1,2) ) / n 
var = (image_f**2).sum((1,2)) / n - avg**2
print( 'std:', torch.sqrt(var) )
print( 'std:', torch.std(image_f, (1,2)) )
# image_normed /= torch.sqrt(var).reshape(-1,1,1)

# print(image_f.std((1,2)).reshape(-1,1,1))
# image_normed = (image_f - image_f.mean((1,2)).reshape(-1,1,1)) / image_f.std((1,2)).reshape(-1,1,1)
image_normed = (image_f - avg.reshape(-1,1,1)) / torch.sqrt(var).reshape(-1,1,1)

print('normed')
for t in image_normed:
    mean, std, var = torch.mean(t), torch.std(t), torch.var(t)
    print(mean, std, std**2, var)

In [ ]:
importlib.reload(funcs)

class Trainer():
    def __init__(self):
        self.i = 0

    def compute(self):
        print('compute', self.i)
        time.sleep(0.5)
        self.i += 1
        return self.i > 30


trainer = Trainer()
funcs.run_with_stop_button(trainer.compute)

In [ ]:
funcs.get_conv_resize(224, 11, 4, 0)
funcs.get_conv_resize(54, 3, 2, 0)
funcs.get_conv_resize(26, 5, 1, 2)
funcs.get_conv_resize(26, 3, 2, 0)
funcs.get_conv_resize(12, 3, 2, 0)
# funcs.get_conv_resize(26, 3, 2, 0)

In [ ]:
import tkinter as tk
import threading
import time

# Function to close the pop-up window
def close_popup():
    global popup_closed
    popup_closed = True
    popup.destroy()

# Worker thread function
def worker(popup_window):
    print('worker')
    for i in range(3):
        if popup_closed:
            break
        # Perform background computations or updates here
        # ...
        print('compute')
        # Sleep for a while to avoid excessive CPU usage
        time.sleep(0.5)  # Adjust sleep duration as needed
    popup_window.after(0, popup_window.destroy())
    global processed
    processed = True
# Create the main window
root = tk.Tk()

# Create the pop-up window
popup_closed = False
processed = False
# popup = tk.Toplevel(root)
popup = root
popup.title("Pop-up Window")

# Add some content to the pop-up window
content_label = tk.Label(popup, text="This is a pop-up window!")
content_label.pack(padx=20, pady=20)

# Add a button to close the pop-up window
close_button = tk.Button(popup, text="Close", command=close_popup)
close_button.pack(pady=10)

# Create and start the worker thread
thread = threading.Thread(target=worker, args=(root,))
thread.start()

# Run the main event loop
root.mainloop()

# Wait for the worker thread to finish
# thread.join()
while not processed:
    print('waiting end process')
    time.sleep(1)


In [ ]:
!pip install -U ipywidgets==7.7.1

In [ ]:
!jupyter nbextension enable --py --sys-prefix widgetsnbextension

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from traitlets import Bool, observe
import time

# Custom pop-up widget
class Popup(widgets.Box):
    closed = Bool(False)

    def __init__(self, content):
        super().__init__()
        self.content = content

        self.close_button = widgets.Button(description="Close")
        self.close_button.on_click(self.close)

        self.children = [self.content, self.close_button]

    def close(self, _):
        print('CLOSE FUNC')
        self.closed = True

# Create the pop-up window
popup = Popup(widgets.HTML('This is a pop-up window!'))
display(popup)

# Run the loop until the pop-up window is closed
while not popup.closed:
    time.sleep(1)
    pass


In [ ]:
import tkinter as tk

# Create the main window
root = tk.Tk()

# Function to close the pop-up window
def close_popup():
    popup.destroy()

# Create the pop-up window
popup = tk.Toplevel(root)
popup.title("Pop-up Window")

# Add some content to the pop-up window
content_label = tk.Label(popup, text="This is a pop-up window!")
content_label.pack(padx=20, pady=20)

# Add a button to close the pop-up window
close_button = tk.Button(popup, text="Close", command=close_popup)
close_button.pack(pady=10)

# Run the main event loop
root.mainloop()


In [ ]:
from tkinter import *
import threading
import time

class Show_Azure_Message(Toplevel):
    def __init__(self,master,message):
        Toplevel.__init__(self,master) #master have to be Toplevel, Tk or subclass of Tk/Toplevel
        self.title('')
        self.attributes('WM_DELETE_WINDOW', self.callback)
        self.resizable(False,False)
        Label(self,text=message,font='-size 25').pack(fill=BOTH,expand=True)
        self.geometry('250x50+%i+%i'%((self.winfo_screenwidth()-250)//2,(self.winfo_screenheight()-50)//2))

    def callback(self): pass

BasicApp=Tk()
App = Show_Azure_Message(BasicApp,'Hello')
for i in range(0,2):
    print(i)
    time.sleep(1)

App.destroy()

In [ ]:
# import hiddenlayer as hl

# batch = next(iter(dataloaders['train']))

# transforms = []
# # transforms = [ hl.transforms.Prune('Constant') ] # Removes Constant nodes from graph.
# graph = hl.build_graph(trainer.model, batch[0].text, transforms=transforms)
# graph.theme = hl.graph.THEMES['blue'].copy()
# graph.save('rnn_hiddenlayer', format='png')

In [ ]:
import hiddenlayer as hl
importlib.reload(funcs)
model = funcs.NeuralNetwork(10, True)
batch = next(iter(dataloader_train))

transforms = []
# transforms = [ hl.transforms.Prune('Constant') ] # Removes Constant nodes from graph.

graph = hl.build_graph(model, batch.text, transforms=transforms)
graph.theme = hl.graph.THEMES['blue'].copy()
graph.save('rnn_hiddenlayer', format='png')

In [ ]:
importlib.reload(funcs)
from torchviz import make_dot
model = funcs.NeuralNetwork(10, True)
dummy_input = torch.randn(2, 1, 224, 224)
output = model(dummy_input)
dot = make_dot(output, params=dict(model.named_parameters()))
dot_string = dot.to_string()

import pydot
graph = pydot.graph_from_dot_data(dot_string)[0]

# dot.format = 'pdf'
# dot.render("model_graph")

plt.figure(figsize=(10, 10))
plt.imshow(graph.create_png(), aspect='equal')
plt.axis('off')
plt.show()

In [ ]:
importlib.reload(funcs)
# funcs.test_plot_slider()

y = np.array([0,0,0,0, 1,1,1,1, 2,2,2,2])
pred = np.array([0,0,0,2, 1,1,1,0, 2,0,1,2])

classes = ['class 1', 'class 2', 'class 3']
cm = confusion_matrix(y, pred)
print(cm)
funcs.create_confusion_plot(cm, classes)


In [ ]:
importlib.reload(funcs)
funcs.plot_schedulers()

In [ ]:
kernel_size = 7
sigma = 3./3
gaussianBlur = transforms.GaussianBlur(kernel_size, sigma)
image = torch.zeros((1, kernel_size,kernel_size), dtype=torch.float)
image[0, kernel_size//2, kernel_size//2] = 1.

# image
gaussianBlur(image).numpy().round(3)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

def visualize_gaussian_kernel(kernel_size, sigma):
    # Create a 2D Gaussian kernel
    kernel = torch.zeros((kernel_size, kernel_size))
    center = kernel_size // 2

    for i in range(kernel_size):
        for j in range(kernel_size):
            x = i - center
            y = j - center
            kernel[i, j] = torch.exp(-(x ** 2 + y ** 2) / (2 * sigma ** 2))
    
    # Normalize the kernel
    kernel = kernel / kernel.sum()

    # Convert the kernel tensor to a numpy array for visualization
    kernel_np = kernel.numpy()

    # Display the kernel using a heatmap
    plt.imshow(kernel_np, cmap='hot', interpolation='nearest')
    plt.title('Gaussian Blur Kernel')
    plt.colorbar()
    plt.show()

# Example usage
kernel_size = 5
sigma = 1.0
visualize_gaussian_kernel(kernel_size, sigma)

In [ ]:
torch.cuda.is_available()

In [ ]:
torch.cuda.device_count()

In [ ]:
torch.cuda.current_device()

In [ ]:
torch.cuda.device(0)

In [ ]:
torch.cuda.get_device_name(0)